In [ ]:
# see link
# https://github.com/Fraud-Detection-Handbook

package ='06-sklearn.neighbors'
name='KNeighborsClassifier'
tuningAndParameters='02-After tuning'

hyperparametersFound = {'n_neighbors':3}
scalerFound='StandardScaler'



print('done')

# Data preparation

In [ ]:
import sys
import os
from importlib import reload
fpath = os.path.join('..//scripts')
sys.path.append(fpath)

import warnings
warnings.filterwarnings('ignore')

#loading internal scripts
import datamanagement as dm
reload(dm)

import result as resultMd
reload(resultMd)

import graph as gf
reload(gf)

import scaler as scaler
reload(scaler)

print('done')

In [ ]:
dfLearning, dfValidation =dm.getDataLearningAndValidation()

dfLearning.head()

# Sampling choice

* Default no sampling

# Scaling choice

In [ ]:
%%script false

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from imblearn.under_sampling import NearMiss
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()
scalingData=[]

scalers = scaler.getScalers()
for key in scalers:
    print(key)
    x1, y1 = dfLearning[predictors], dfLearning[target]
    sc=scalers.get(key)
    x2 = sc.fit_transform(x1)

    TEST_SIZE = 0.20 # test size using_train_test_split
    RANDOM_STATE = 0


    x_train0, x_test, y_train0, y_test = train_test_split(x2, y1, test_size = TEST_SIZE, 
                                                        stratify=y1,
                                                        random_state = RANDOM_STATE)


    
    modelClf = KNeighborsClassifier()
    parameters=hyperparametersFound
    modelClf.set_params(**parameters)

    modelClf.fit(x_train0, y_train0)
    predsTrain = modelClf.predict(x_train0)
    predsTest = modelClf.predict(x_test)

    train_f1=f1_score(y_train0, predsTrain)
    print("f1 train {:.4f}".format(train_f1))
    
    test_f1=f1_score(y_test, predsTest)
    print("f1 test  {:.4f}".format(test_f1))
    print('-----------------------')
    subScalingData = [train_f1,test_f1]
    scalingData.append(subScalingData)

print(scalingData)
import matplotlib.pyplot as plt

fig = plt.figure(figsize =(10, 7))
ax = fig.add_subplot(111)
bp = ax.boxplot(scalingData, patch_artist = True,
                notch ='True', vert = 0)
ax.set_yticklabels(['StandardScaler','MinMaxScaler','RobustScaler','MaxAbsScaler'])
plt.title("Scaling choice")
ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()
plt.show()

# Hyperparameters tuning

## Grid Search
* grid Search only
    * because the search time is quite short
    * because there is only one hyper parameter

In [ ]:


from imblearn.under_sampling import NearMiss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

x1, y1 = dfLearning[predictors], dfLearning[target]
sc = scaler.getScaler(scalerFound)
x2 = sc.fit_transform(x1)


x_train, x_test, y_train, y_test = train_test_split(x2, y1, test_size = TEST_SIZE, 
                                                        stratify=y1,
                                                        random_state = RANDOM_STATE)

In [ ]:
%%script false

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
import numpy as np 


dic_param={
    'n_neighbors':np.arange(1 , 13)
}
modelClf = KNeighborsClassifier()

random_search = GridSearchCV(modelClf,dic_param, scoring='f1', verbose=10,cv=4).fit(x_train, y_train)
print(random_search.best_params_)
print(random_search.best_score_)


#{'n_neighbors': 3}
#0.6712944235982508


# Model evaluation

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from datetime import datetime

modelClf = KNeighborsClassifier()
parameters=hyperparametersFound

modelClf.set_params(**parameters)
then= datetime.now()
modelClf.fit(x_train, y_train)
now = datetime.now()
duration= now - then
learningDurationInS = duration.total_seconds()
print("Duration ",learningDurationInS )
resultMd.update_time_response_result(package, name, tuningAndParameters, learningDurationInS)

predsTrain = modelClf.predict(x_train)
predsTest = modelClf.predict(x_test)

f1Learning =f1_score(y_train, predsTrain)
f1Test=f1_score(y_test, predsTest)
dm.show_confusion_matrix(y_train, predsTrain,'Confusion matrix learning data')
print(f"f1 train {f1Learning:.3f}")
dm.show_confusion_matrix(y_test, predsTest,'Confusion matrix test data')
print(f"f1 test {f1Test:.3f}")
resultMd.update_learning_test_result(package, name, tuningAndParameters, f1Learning,f1Test)

In [ ]:
gf.show_importance(modelClf, predictors)

In [ ]:
gf.show_prediction_graph(modelClf, x_test,y_test)

In [ ]:
dfValidatationScaled = sc.transform(dfValidation[predictors])

predsValidation = modelClf.predict(dfValidatationScaled)
f1Validation=f1_score(dfValidation[target], predsValidation)

dm.show_confusion_matrix(dfValidation[target], predsValidation,'Confusion matrix validation data')
print(f"f1 validation {f1Validation:.3f}")
resultMd.update_performance_result(package, name, tuningAndParameters, f1Validation)
resultMd.update_hyperparameters_result(package, name, tuningAndParameters, hyperparametersFound,scalerFound)


# Summary

In [ ]:
print('Summary')
print(f"{package} {name} {tuningAndParameters}") 
print(f"hyperparameters {hyperparametersFound}") 
print(f"scaler {scalerFound}") 
print('-----------------------------')

print(f"learning duration {learningDurationInS:.2f} s")
print('-----------------------------')
print(f"f1 train      {f1Learning:.3f}")
print(f"f1 test       {f1Test:.3f}")
print(f"f1 validation {f1Validation:.3f}")